Now that we fitted the ANN in experiments.ipynb, what if we have a new sample that we want to feed into the ANN?

In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
### Load the trained model, scaler pickle,onehot
model = load_model('model.h5')

## load the encoder and scaler
with open('onehot_encoder_geo.pk1','rb') as file: # rb => read by
	onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pk1','rb') as file:
	label_encoder_gender = pickle.load(file)

with open('scaler.pk1','rb') as file:
	scaler = pickle.load(file)

In [ ]:
# Example input data

input_data = {
'CreditScore': 600,
'Geography': 'France',
'Gender':'Male',
'Age':40,
'Tenure':3,
'Balance':60000,
'NumOfProducts':2,
'HasCrCard':1,
'IsActiveMember':1,
'EstimatedSalary':50000
}

## He challenges us at this point to try to convert this input data into a form that we can feed into the model.
# How do we encode Geography, Gender.. and StandardScaler application?

# I presume we should use the pickled settings as we imported previously, on the relevant columns

In [7]:
input_data['Gender'] = label_encoder_gender.transform(pd.DataFrame(input_data['Gender']))
input_data['Geography'] = onehot_encoder_geo.transform(pd.DataFrame(input_data['Geography']))

## It is slightly less clear to me what I should do now. See, StandardScaler is fitted on the train data,
# and then used to transform subsequent data. Here, it would be fine to just use it for transformation.
# However, I need to adjust input_data as a single observation, in the same way that I adjusted the entire collection, right?

# By that I mean, I need to make a DataFrame of the Geography onehot. Make input_data a DataFrame (minus the Geography column), right?
# Or is this not actually the case? Surely now that I've transformed the original column, this will do the trick?

# In Krish's case, he transformed the one column into three indicator columns. But I should see what happens when I run this code.

ValueError: DataFrame constructor not properly called!

I suppose the fundamental issue here is that I'm trying to turn a single value into a DataFrame. And without turning it into a dataframe, it does not appreciate the argument to have any dimension. Only a shape of ()...


In [16]:
test = pd.DataFrame(input_data.values(),columns=input_data.keys())

ValueError: Shape of passed values is (10, 1), indices imply (10, 10)

Proceeding with his code:

In [20]:
## One-hot encode 'Geography'

geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]) # need to use a second set of brackets. This threw him at first, too.
# His explanation as to how we should know to use two sets of brackets is... well. shit. All he says is that the input needs to be a list of features. bruh
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

c:\Users\danwh\Documents\Udemy Courses\Krish Naik Data Science, Machine Learning\End to End DL Project using ANN\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [21]:
# Convert dictionary into dataframe
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [22]:
# Combine one-hot encoded columns with input data
input_data = pd.concat([input_df.reset_index(drop=True), geo_encoded_df], axis=1)
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [23]:
# Encode categorical variables
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [24]:
## Concatenation with one-hot encoded data

#(I note that his approach of doing it one-at-a-time kind of sucks?)

input_df = pd.concat([input_df.drop('Geography',axis=1),geo_encoded_df],axis=1)

input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [25]:
## Finally, need to scale the input data

input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [26]:
## Predict churn (whether the person will leave the bank or not)

prediction = model.predict(input_scaled)

prediction_probability = prediction[0][0]

if prediction_probability > 0.5:
	print('The customer is likely to churn.')
else:
	print('The customer is not likely to churn.')


1/1 [==============================] - 0s 433ms/step
The customer is not likely to churn.


In [ ]:
prediction_probability

0.041487116